# 22 - Delivery
Run the setup cell first, then the work cells in order. Use the WSL mlop-pj kernel.
These cells change cloud resources when you run them. A pending result needs a wait and a phase retry, not a new job.
Use notebooks 18-22 for the five main parts. Training and log waits can overlap with GitHub setup. See docs/RUN_FINISH.md.


In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert Path(sys.prefix).name == "mlop-pj", "Select the WSL mlop-pj kernel."
assert "microsoft" in os.uname().release.lower(), "Open this project in WSL."
status = json.loads((ROOT / "reports/azure-blob-upload.json").read_text())
assert status.get("data_uploaded") and status.get("all_hashes_match") and not status["errors"], "Finish 17 first."

def step(phase):
    print("Running:", phase, flush=True)
    process = subprocess.run([sys.executable, "-m", "cloudlayer.finish", phase],
                             cwd=ROOT, capture_output=True, text=True)
    if process.returncode:
        raise RuntimeError("Phase process failed. Check its stderr before retrying: " + process.stderr)
    result = json.loads(process.stdout)
    print(json.dumps(result, indent=2))
    if result["errors"]:
        raise RuntimeError(f"Check reports/finish-{phase}.json and the matching log in tmp. Stop this group.")
    if result.get("pending"):
        print("Still waiting. Read next in the report; do not recreate the resource or job.")
    return result

print("Root:", ROOT)
print("17 passed. Run this notebook work cells in order.")

Root: /mnt/c/Users/SN23341J001185/Documents/MyWork/ICT/ICTYear4_1/MLop/public_teaching_mlaiops/course/MLOP-PJ
17 passed. Run this notebook work cells in order.


## Group 5 — Final checks and delivery
After the training job and both GitHub runs finish, run this cell. It refreshes registry, alert, and CI evidence, writes the one-page model card, checks provider separation, and queries cost per 1,000 observed predictions.

`pending` means evidence is still missing. Billing can arrive later, so a planning estimate is kept separate from the reported cost. Read `docs/DELIVERY_STATUS.md` and `reports/finish-delivery.json` for the remaining items. Never replace pending results with a claimed pass.

If `gh` is missing, complete its setup in the guide before this cell. The code will stop on a real command error; rerun the relevant phase rather than Group 1.

In [2]:
registry = step("training-registry")
assert registry.get("registry_ready"), "Wait for registry verification before continuing."


Running: training-registry
{
  "checked_at": "2026-10-11T18:03:48.008925+00:00",
  "phase": "training-registry",
  "errors": [
    "RuntimeError: Azure CLI failed: UserError, BadArgument, ArgumentOutOfRange, ModelVersionOutOfRange"
  ],
  "job_name": "train-20261011-164656",
  "status": "Completed",
  "registry_ready": false
}


RuntimeError: Check reports/finish-training-registry.json and the matching log in tmp. Stop this group.

### Alert evidence
Refresh the real alert record. No new failure is injected.


In [ ]:
alert = step("alert")


### CI/CD evidence
Run after GitHub setup, the reviewed push, and both workflow runs. gh must be installed and logged in. If this fails, rerun only this cell after fixing GitHub; do not repeat registry.


In [ ]:
ci_evidence = step("ci-evidence")


### Delivery report
Run after the evidence cells. Read open_items and the separate latency/cold-start flags. A report file alone does not mean the project is ready to submit.


In [ ]:
delivery = step("delivery")


## Cleanup after saving evidence
Save the reports, workflow links, dashboard and alert screenshots, model card, training run, and registry lineage first. See `docs/RUN_FINISH.md` for the exact cleanup command. Cleanup deletes the whole project resource group, including uploaded data and models. Run it yourself only when those resources are no longer needed.

The reminder is 13 October 2026, 21:15 Bangkok time. Verify `reports/cleanup.json` after deletion. There is no automatic cleanup.